In [1]:
# ======================================================
# NOTEBOOK: PROCESS "IN SEARCH OF THE CASTAWAYS" (H100 VERSION)
# ======================================================

In [2]:
!pip install -U "transformers>=4.40.0" accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 30.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [3]:
import os
import re
import pandas as pd
import pickle
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm import tqdm

In [4]:
# --- 1. CONFIG ---
BOOK_NAME = "In Search of the Castaways"
FILENAME = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/In search of the castaways.txt" 
OUTPUT_PKL = "castaways_chunks.pkl"
OUTPUT_EMB = "castaways_embeddings_qwen.pkl"

# SOTA Model for H100 (Instruction-Aware)
# Using gte-Qwen2-7B-instruct
MODEL_ID = "Alibaba-NLP/gte-Qwen2-7B-instruct" 

CHUNK_SIZE = 600 # Words (~800-900 tokens)
OVERLAP = 150

# Major entities to tag
CHARACTERS = [
    "Glenarvan", "Helena", "Grant", "Mary", "Robert", "Paganel", 
    "MacNabb", "John Mangles", "Ayrton", "Thalcave", "Mulrady", "Wilson", "Olbinett"
]

In [5]:
# --- 2. MODEL HELPER FUNCTIONS ---

def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Pooling strategy for Qwen/GTE models.
    Pulls the embedding from the last valid token (end of sequence).
    """
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

def get_embeddings(texts, model, tokenizer, batch_size=4):
    """
    Generates embeddings using the Instruction-Aware model.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize
        batch_dict = tokenizer(
            batch_texts, 
            max_length=8192, 
            padding=True, 
            truncation=True, 
            return_tensors='pt'
        ).to("cuda")

        with torch.no_grad():
            # Pass use_cache=False explicitly
            outputs = model(**batch_dict, use_cache=False)
            
            embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
            embeddings = F.normalize(embeddings, p=2, dim=1)
            
        all_embeddings.append(embeddings.cpu().numpy())
        
    # Flatten list of arrays
    return [item for sublist in all_embeddings for item in sublist]


In [6]:
# --- 3. CLEANING & CHUNKING LOGIC ---

def clean_castaways_text(text):
    print("Cleaning text...")
    # 1. Start Marker (Specific to this book)
    start_match = re.search(r"CHAPTER I\.\s+THE SHARK", text)
    if start_match:
        text = text[start_match.start():]
        print("-> Found Start of Chapter I.")
    else:
        print("-> WARNING: Start marker not found!")
    
    # 2. End Marker (Gutenberg Footer)
    end_match = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK", text)
    if end_match:
        text = text[:end_match.start()]
        print("-> Found End Marker.")
        
    # 3. Noise Removal
    text = re.sub(r"\[Illustration.*?\]", "", text, flags=re.DOTALL)
    text = re.sub(r"\[Sidenote:.*?\]", "", text)
    return text

def get_metadata(chunk_text):
    found = [c for c in CHARACTERS if c in chunk_text]
    if found: return f"Characters: {', '.join(found)}"
    return ""

def process_book(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"File {filename} not found! Upload it first.")
        
    with open(filename, 'r', encoding='utf-8') as f:
        raw_text = f.read()
        
    clean_text = clean_castaways_text(raw_text)
    words = clean_text.split()
    chunks = []
    
    # Regex to track Chapter Headers
    chapter_pattern = re.compile(r"^CHAPTER\s+[IVXLCDM]+\.?")
    current_chapter = "CHAPTER I"
    
    print("Chunking...")
    for i in range(0, len(words), CHUNK_SIZE - OVERLAP):
        chunk_words = words[i:i + CHUNK_SIZE]
        chunk_text = " ".join(chunk_words)
        
        # Check for chapter change
        start_snippet = " ".join(chunk_words[:10])
        match = chapter_pattern.search(start_snippet)
        if match:
            current_chapter = match.group(0)
            
        meta = get_metadata(chunk_text)
        
        # SEARCH TEXT: Combine Meta + Content
        # We assume the index side doesn't need "Instruct:" prefix for GTE/Qwen
        # (It's usually added only on the query side, or "passage: " prefix depending on exact model docs.
        # For Alibaba GTE, raw text is standard for documents).
        search_text = f"{meta} {chunk_text}" 
        
        chunks.append({
            "text": chunk_text,
            "book_name": BOOK_NAME,
            "chapter": current_chapter,
            "metadata": meta,
            "search_text": search_text, 
            "chunk_index": i
        })
        
    return chunks


In [7]:
# --- 4. EXECUTION ---

# A. Load Model (Robust Version)
print(f"Loading SOTA Embedding Model: {MODEL_ID}...")
from transformers import AutoConfig

try:
    # 1. Load Config & Disable Cache (Fixes DynamicCache error)
    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config.use_cache = False 

    # 2. Initialize Tokenizer & Model globally
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        MODEL_ID, 
        config=config,
        trust_remote_code=True,
        torch_dtype=torch.float16 # Faster on GPU
    ).to("cuda")
    model.eval()
    
    print("Model and Tokenizer loaded successfully.")

except Exception as e:
    print(f"CRITICAL ERROR loading model: {e}")
    # Stop execution if model fails
    raise e

Loading SOTA Embedding Model: Alibaba-NLP/gte-Qwen2-7B-instruct...


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00002-of-00007.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00007-of-00007.safetensors:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

model-00004-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully.


In [8]:
# B. Process Text
print(f"Processing {FILENAME}...")
data = process_book(FILENAME)
df = pd.DataFrame(data)
print(f"Generated {len(df)} chunks.")

Processing /kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/Books/In search of the castaways.txt...
Cleaning text...
-> Found Start of Chapter I.
-> Found End Marker.
Chunking...
Generated 299 chunks.


In [9]:
# C. Embed (Now tokenizer is guaranteed to exist)
print("Generating Embeddings...")
embeddings = get_embeddings(df['search_text'].tolist(), model, tokenizer)

Generating Embeddings...


Embedding: 100%|██████████| 75/75 [00:09<00:00,  8.18it/s]


In [10]:
# D. Save
df.to_pickle(OUTPUT_PKL)
with open(OUTPUT_EMB, "wb") as f:
    pickle.dump(embeddings, f)

print(f"DONE! Saved {OUTPUT_PKL} and {OUTPUT_EMB}")

DONE! Saved castaways_chunks.pkl and castaways_embeddings_qwen.pkl
